In [21]:
import torch
from torch.utils.data import DataLoader, TensorDataset

In [22]:
# Cell 2
class DataModule:
    """훈련 및 validation data loader를 제공하는 base class."""

    def __init__(self, batch_size=2, num_workers=0):
        # 한 번에 모델에 전달할 sample 수
        self.batch_size = batch_size

        # 데이터를 불러오는 별도 worker process 수
        # Notebook과 WSL에서는 우선 0으로 두는 것이 안정적이다.
        self.num_workers = num_workers

    def get_dataloader(self, train):
        # 실제 data loader 생성 방식은 자식 class가 정의한다.
        raise NotImplementedError

    def train_dataloader(self):
        # train=True를 전달하여 훈련 data loader를 가져온다.
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        # train=False를 전달하여 validation data loader를 가져온다.
        return self.get_dataloader(train=False)

In [23]:
# Cell 3
class RegressionData(DataModule):
    
    def __init__(self, batch_size=2, num_workers=0):
        # 부모 class가 batch_size와 num_workers를 저장한다.
        super().__init__(
            batch_size=batch_size,
            num_workers=num_workers,
        )

        # sample 6개, feature 2개
        features = torch.tensor([
            [1.0, 2.0],
            [2.0, 1.0],
            [3.0, 4.0],
            [4.0, 3.0],
            [5.0, 2.0],
            [2.0, 5.0],
        ])

        # 각 sample에 대응하는 label
        labels = torch.tensor([
            [1.0],
            [4.0],
            [-1.0],
            [2.0],
            [13.0],
            [-6.0],
        ])
        
        self.train_dataset = TensorDataset(
            features[:4],
            labels[:4],
        )
        
        self.val_dataset = TensorDataset(
            features[4:],
            labels[4:],
        )
        
        
    def get_dataloader(self, train):
        # train=True면 훈련용, False면 validation용 데이터셋을 선택한다.
        if train:
            dataset = self.train_dataset
        else:
            dataset = self.val_dataset
            
        return DataLoader(
            dataset=dataset,
            batch_size=self.batch_size,
            shuffle=train,
            num_workers=self.num_workers,
        )


In [24]:
# Cell 4
data = RegressionData(batch_size=2)

train_loader = data.train_dataloader()
val_loader = data.val_dataloader()

print("Number of training samples:", len(data.train_dataset))
print("Number of validation samples:", len(data.val_dataset))

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))

Number of training samples: 4
Number of validation samples: 2
Number of training batches: 2
Number of validation batches: 1


In [25]:
# Cell 5
# DataLoader를 반복하면 매번 sample 2개로 구성된 batch가 나온다.
for batch_index, (batch_features, batch_labels) in enumerate(train_loader):
    print(f"Batch {batch_index + 1}")
    print("Features:")
    print(batch_features)
    print("Features shape:", batch_features.shape)
    print("Labels:")
    print(batch_labels)
    print("Labels shape:", batch_labels.shape)
    print()

    assert batch_features.shape == (2, 2)
    assert batch_labels.shape == (2, 1)

Batch 1
Features:
tensor([[2., 1.],
        [4., 3.]])
Features shape: torch.Size([2, 2])
Labels:
tensor([[4.],
        [2.]])
Labels shape: torch.Size([2, 1])

Batch 2
Features:
tensor([[1., 2.],
        [3., 4.]])
Features shape: torch.Size([2, 2])
Labels:
tensor([[ 1.],
        [-1.]])
Labels shape: torch.Size([2, 1])



In [26]:
# Cell 6
# iter()는 DataLoader를 batch iterator로 바꾸고,
# next()는 그중 첫 번째 batch 하나를 꺼낸다.
train_iterator = iter(train_loader)
first_features, first_labels = next(train_iterator)

print("First batch features:")
print(first_features)
print("First batch labels:")
print(first_labels)

assert first_features.shape == (2, 2)
assert first_labels.shape == (2, 1)

First batch features:
tensor([[1., 2.],
        [4., 3.]])
First batch labels:
tensor([[1.],
        [2.]])
